# Data Quality — Bronze Layer Gate

Runs all DQ rule sets against every **Bronze** table.
- ✅ **Passing rows** flow through to Silver unchanged.
- ❌ **Failing rows** are written to `hospital_catalog.quarantine.*`.
- 📊 All results are logged to `hospital_catalog.quality.dq_results`.

**Job position:** runs immediately after `01_Ingest_Bronze`, before `02_Process_Silver`.

| Step | What happens |
|------|--------------|
| 1 | Environment setup |
| 2 | Schema / table bootstrap (idempotent) |
| 3 | Table configuration |
| 4 | Execute DQ checks → quarantine bad rows |
| 5 | Summary — failure rates & critical alerts |

## 1. Setup

In [ ]:
import sys
from datetime import datetime

sys.path.insert(0, '/Workspace/Users/sallam5100@gmail.com/.bundle/hospital_edw/dev/files')
print('Environment ready.')

In [ ]:
# Optional: limit to a single table via job base_parameters or interactive widget.
# Leave as 'ALL' to check every Bronze table.
try:
    dbutils.widgets.text('TABLE_FILTER', 'ALL', 'Table filter (key or ALL)')
    TABLE_FILTER = dbutils.widgets.get('TABLE_FILTER')
except Exception:
    TABLE_FILTER = 'ALL'

print(f'TABLE_FILTER = {TABLE_FILTER!r}')

In [ ]:
from config.quality_config import (
    DQ_CATALOG,
    QUALITY_SCHEMA,
    QUARANTINE_SCHEMA,
    BRONZE_SCHEMA,
    DQ_RESULTS_TABLE,
    DQ_RUNS_TABLE,
)

from src.quality.engine import run_dq

from src.quality.rules import (
    PATIENT_RULES,
    DOCTOR_RULES,
    DEPARTMENT_RULES,
    ENCOUNTER_RULES,
    ADMISSION_RULES,
    DIAGNOSIS_RULES,
    PROCEDURE_RULES,
    LAB_ORDER_RULES,
    LAB_RESULT_RULES,
    MEDICATION_RULES,
    PRESCRIPTION_RULES,
    PHARMACY_TRANSACTION_RULES,
    INVOICE_RULES,
    PAYMENT_RULES,
    INSURANCE_CLAIM_RULES,
)

print('All imports successful.')

## 2. Schema and Table Bootstrap

Creates `quality` and `quarantine` schemas and their tracking tables if they do not already exist. Safe to re-run.

In [ ]:
bootstrap_statements = [
    f'CREATE SCHEMA IF NOT EXISTS {DQ_CATALOG}.{QUALITY_SCHEMA}',
    f'CREATE SCHEMA IF NOT EXISTS {DQ_CATALOG}.{QUARANTINE_SCHEMA}',
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RESULTS_TABLE} (
        dq_run_id        STRING    NOT NULL,
        dq_timestamp     TIMESTAMP NOT NULL,
        table_name       STRING    NOT NULL,
        source_branch    STRING,
        source_system    STRING,
        rule_id          STRING    NOT NULL,
        rule_name        STRING    NOT NULL,
        dq_category      STRING    NOT NULL,
        severity         STRING    NOT NULL,
        column_name      STRING,
        total_records    BIGINT    NOT NULL,
        failed_records   BIGINT    NOT NULL,
        passed_records   BIGINT    NOT NULL,
        failure_rate     DOUBLE,
        status           STRING    NOT NULL,
        rule_description STRING
    ) USING DELTA
    """,
    f"""
    CREATE TABLE IF NOT EXISTS {DQ_RUNS_TABLE} (
        dq_run_id         STRING    NOT NULL,
        start_timestamp   TIMESTAMP NOT NULL,
        end_timestamp     TIMESTAMP,
        run_status        STRING,
        total_tables      BIGINT,
        total_rules       BIGINT,
        failed_rules      BIGINT,
        critical_failures BIGINT,
        total_records     BIGINT,
        failed_records    BIGINT,
        created_at        TIMESTAMP
    ) USING DELTA
    """,
]

for stmt in bootstrap_statements:
    spark.sql(stmt)

print(f'  OK  {DQ_CATALOG}.{QUALITY_SCHEMA}')
print(f'  OK  {DQ_CATALOG}.{QUARANTINE_SCHEMA}')
print(f'  OK  {DQ_RESULTS_TABLE}')
print(f'  OK  {DQ_RUNS_TABLE}')
print('Bootstrap complete.')

## 3. Bronze Table Configuration

Maps each Bronze table to its DQ rule set and source system label.
Set `TABLE_FILTER` to a specific key (e.g. `'patients'`) to run a single table,
or leave as `'ALL'` to process everything.

In [ ]:
BRONZE_DQ_CONFIG = {

    # ── EMR ──────────────────────────────────────────────────────────────
    'patients': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.patients',
        'rules':         PATIENT_RULES,
        'source_system': 'EMR',
    },
    'doctors': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.doctors',
        'rules':         DOCTOR_RULES,
        'source_system': 'EMR',
    },
    'departments': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.departments',
        'rules':         DEPARTMENT_RULES,
        'source_system': 'EMR',
    },
    'encounters': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.encounters',
        'rules':         ENCOUNTER_RULES,
        'source_system': 'EMR',
    },
    'admissions': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.admissions',
        'rules':         ADMISSION_RULES,
        'source_system': 'EMR',
    },
    'diagnoses': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.diagnoses',
        'rules':         DIAGNOSIS_RULES,
        'source_system': 'EMR',
    },
    'procedures': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.procedures',
        'rules':         PROCEDURE_RULES,
        'source_system': 'EMR',
    },

    # ── Laboratory ────────────────────────────────────────────────────────
    'lab_orders': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.lab_orders',
        'rules':         LAB_ORDER_RULES,
        'source_system': 'LABORATORY',
    },
    'lab_results': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.lab_results',
        'rules':         LAB_RESULT_RULES,
        'source_system': 'LABORATORY',
    },

    # ── Pharmacy ──────────────────────────────────────────────────────────
    'medications': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.medications',
        'rules':         MEDICATION_RULES,
        'source_system': 'PHARMACY',
    },
    'prescriptions': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.prescriptions',
        'rules':         PRESCRIPTION_RULES,
        'source_system': 'PHARMACY',
    },
    'pharmacy_transactions': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.pharmacy_transactions',
        'rules':         PHARMACY_TRANSACTION_RULES,
        'source_system': 'PHARMACY',
    },

    # ── Billing ───────────────────────────────────────────────────────────
    'invoices': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.invoices',
        'rules':         INVOICE_RULES,
        'source_system': 'BILLING',
    },
    'payments': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.payments',
        'rules':         PAYMENT_RULES,
        'source_system': 'BILLING',
    },
    'insurance_claims': {
        'table':         f'{DQ_CATALOG}.{BRONZE_SCHEMA}.insurance_claims',
        'rules':         INSURANCE_CLAIM_RULES,
        'source_system': 'BILLING',
    },
}

active = (
    BRONZE_DQ_CONFIG
    if TABLE_FILTER == 'ALL'
    else {TABLE_FILTER: BRONZE_DQ_CONFIG[TABLE_FILTER]}
)

print(f'Running DQ on {len(active)} Bronze tables (TABLE_FILTER={TABLE_FILTER!r})')
for k in active:
    print(f'  → {k}')

## 4. Execute DQ Checks

For each table:
1. Every rule is evaluated against the full Bronze table.
2. **Failing rows** are written to `hospital_catalog.quarantine.<table>`.
3. **All results** (pass/fail counts, failure rates) are appended to `dq_results`.

The Silver processing step reads from Bronze directly — Silver transforms are designed
to be robust, and quarantined rows are simply noted in the audit trail.
CRITICAL failures raise an error to block the job from proceeding.

In [ ]:
from pyspark.sql import functions as F

all_results        = []
total_failed_rules = 0

for tbl_key, cfg in active.items():
    tbl_path = cfg['table']

    df = spark.table(tbl_path)

    result = run_dq(
        spark,
        df,
        table_name    = tbl_key,
        rules         = cfg['rules'],
        source_system = cfg['source_system'],
    )

    icon = '✅' if result['failed_rules'] == 0 else '❌'
    print(
        f"  {icon}  {tbl_key:<30} "
        f"{result['total_records']:>9,} rows  "
        f"{result['rules_executed']:>3} rules  "
        f"{result['failed_rules']:>3} failed"
    )
    all_results.append(result)
    total_failed_rules += result['failed_rules']

print(f'\nDQ complete — {total_failed_rules} failed rules across {len(all_results)} tables.')

## 5. Summary

In [ ]:
# ── Summary by source system, severity, status ────────────────────────────
display(
    spark.table(DQ_RESULTS_TABLE)
    .groupBy('source_system', 'severity', 'status')
    .agg(
        F.count('*').alias('rule_count'),
        F.sum('failed_records').alias('total_failed_records'),
        F.round(F.avg('failure_rate') * 100, 2).alias('avg_failure_pct'),
    )
    .orderBy('source_system', 'severity')
)

In [ ]:
# ── CRITICAL failures ─────────────────────────────────────────────────────
critical_fails = (
    spark.table(DQ_RESULTS_TABLE)
    .filter((F.col('severity') == 'CRITICAL') & (F.col('status') == 'FAILED'))
    .orderBy(F.desc('failed_records'))
)

n_critical = critical_fails.count()
if n_critical == 0:
    print('✅  No CRITICAL failures — pipeline can proceed to Silver.')
else:
    print(f'⚠️  {n_critical} CRITICAL rule failures logged to quarantine.')
    display(critical_fails.select(
        'table_name', 'rule_id', 'rule_name', 'column_name',
        'total_records', 'failed_records', 'failure_rate', 'rule_description'
    ))

In [ ]:
# ── Quarantine summary — how many rows were quarantined per table ─────────
from pyspark.sql.utils import AnalysisException

quarantine_summary = []
for tbl_key in active:
    qtbl = f'{DQ_CATALOG}.{QUARANTINE_SCHEMA}.{tbl_key}'
    parts = qtbl.split('.')
    exists = spark.catalog.tableExists(f'{parts[1]}.{parts[2]}', parts[0])
    if exists:
        cnt = spark.table(qtbl).count()
        quarantine_summary.append({'table': tbl_key, 'quarantined_rows': cnt})

if quarantine_summary:
    import pandas as pd
    print('Quarantined rows:')
    display(spark.createDataFrame(quarantine_summary))
else:
    print('✅  No rows quarantined in this run.')